# HW4: PubMed RCT text classification (CPU-friendly)

Pipeline: zero-shot baseline, full finetune, linear probe, and LoRA on `armanc/pubmed-rct20k`.
Model: `epfl-llm/MedDistilBERT` with fallback to `distilbert-base-uncased`. Train subset: 30k for CPU practicality (seed 42).


In [ ]:
# If packages are missing, uncomment:
# pip install transformers datasets evaluate peft accelerate


In [ ]:
import os
import random
import time
from pathlib import Path

import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)
import evaluate

try:
    from peft import LoraConfig, TaskType, get_peft_model
except Exception:
    LoraConfig = None
    TaskType = None
    get_peft_model = None

set_seed(42)
np.random.seed(42)
random.seed(42)
print(f"Torch device: {'mps' if torch.backends.mps.is_available() else 'cpu'}")


In [ ]:
cfg = {
    "dataset_name": "armanc/pubmed-rct20k",
    "model_name": "epfl-llm/MedDistilBERT",
    "fallback_model_name": "distilbert-base-uncased",
    "seed": 42,
    "max_length": 256,
    "train_subset": 30_000,
    "batch_size": 8,
    "eval_batch_size": 16,
    "num_epochs_full": 2,
    "num_epochs_probe": 2,
    "num_epochs_lora": 2,
    "warmup_ratio": 0.1,
    "weight_decay": 0.01,
    "lora_r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
}


In [ ]:
raw_ds = load_dataset(cfg["dataset_name"])
print(raw_ds)

label_col = "label"
string_cols = [
    c
    for c in raw_ds["train"].column_names
    if c != label_col and getattr(raw_ds["train"].features[c], "dtype", None) == "string"
]
if not string_cols:
    raise ValueError("No text column detected")
text_col = string_cols[0]
print(f"Using text column: {text_col}")

label_feature = raw_ds["train"].features[label_col]
if hasattr(label_feature, "names") and label_feature.names:
    label_names = list(label_feature.names)
else:
    label_names = sorted(raw_ds["train"].unique(label_col))
    raw_ds = raw_ds.map(lambda x: {"label": label_names.index(x[label_col])})

label2id = {name: i for i, name in enumerate(label_names)}
id2label = {i: name for i, name in enumerate(label_names)}
print(f"Labels: {id2label}")

split_val = "validation" if "validation" in raw_ds else "val"
split_test = "test" if "test" in raw_ds else split_val


In [ ]:
train_ds = raw_ds["train"].shuffle(seed=cfg["seed"]).select(range(cfg["train_subset"]))
val_ds = raw_ds[split_val]
test_ds = raw_ds[split_test]

print(train_ds)
print(val_ds)
print(test_ds)


In [ ]:
def load_tok_and_model(primary, fallback):
    last_error = None
    for name in [primary, fallback]:
        try:
            tok = AutoTokenizer.from_pretrained(name, use_fast=True)
            model = AutoModelForSequenceClassification.from_pretrained(
                name,
                num_labels=len(label_names),
                id2label=id2label,
                label2id=label2id,
            )
            print(f"Loaded model {name}")
            return tok, model, name
        except Exception as exc:
            print(f"Failed to load {name}: {exc}")
            last_error = exc
    raise RuntimeError(f"Could not load any model. Last error: {last_error}")


tokenizer, base_model, actual_model_name = load_tok_and_model(
    cfg["model_name"], cfg["fallback_model_name"]
)


In [ ]:
def preprocess(batch):
    return tokenizer(batch[text_col], truncation=True, max_length=cfg["max_length"])


remove_cols = [c for c in train_ds.column_names if c != label_col]
tokenized = {
    "train": train_ds.map(preprocess, batched=True, remove_columns=remove_cols),
    split_val: val_ds.map(preprocess, batched=True, remove_columns=remove_cols),
    split_test: test_ds.map(preprocess, batched=True, remove_columns=remove_cols),
}

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [ ]:
acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": acc_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"],
    }


def count_trainable_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable params: {trainable/1e6:.2f}M / {total/1e6:.2f}M")
    return trainable, total


In [ ]:
zero_args = TrainingArguments(
    output_dir="outputs/zero_shot",
    per_device_eval_batch_size=cfg["eval_batch_size"],
    seed=cfg["seed"],
    report_to="none",
)

zero_trainer = Trainer(
    model=base_model,
    args=zero_args,
    eval_dataset=tokenized[split_val],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

zero_metrics = zero_trainer.evaluate()
print("Zero-shot metrics:", zero_metrics)
results = {"zero_shot": zero_metrics}


In [ ]:
def fresh_model():
    return AutoModelForSequenceClassification.from_pretrained(
        actual_model_name,
        num_labels=len(label_names),
        id2label=id2label,
        label2id=label2id,
    )


def run_training(tag, model, train_dataset, eval_dataset, training_args):
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    start = time.time()
    trainer.train()
    wall = time.time() - start
    eval_metrics = trainer.evaluate(eval_dataset)
    test_metrics = trainer.evaluate(tokenized[split_test], metric_key_prefix="test")
    metrics = {**eval_metrics, **test_metrics, "wall_time_sec": wall}
    results[tag] = metrics
    print(f"Run {tag} metrics:", metrics)
    return trainer, metrics


In [ ]:
full_args = TrainingArguments(
    output_dir="outputs/full_finetune",
    learning_rate=2e-5,
    per_device_train_batch_size=cfg["batch_size"],
    per_device_eval_batch_size=cfg["eval_batch_size"],
    num_train_epochs=cfg["num_epochs_full"],
    weight_decay=cfg["weight_decay"],
    evaluation_strategy="epoch",
    save_strategy="no",
    logging_steps=100,
    warmup_ratio=cfg["warmup_ratio"],
    seed=cfg["seed"],
    report_to="none",
)

full_model = fresh_model()
count_trainable_params(full_model)
full_trainer, full_metrics = run_training(
    "full_finetune",
    full_model,
    tokenized["train"],
    tokenized[split_val],
    full_args,
)


In [ ]:
probe_args = TrainingArguments(
    output_dir="outputs/linear_probe",
    learning_rate=1e-3,
    per_device_train_batch_size=cfg["batch_size"],
    per_device_eval_batch_size=cfg["eval_batch_size"],
    num_train_epochs=cfg["num_epochs_probe"],
    weight_decay=0.0,
    evaluation_strategy="epoch",
    save_strategy="no",
    logging_steps=100,
    seed=cfg["seed"],
    report_to="none",
)

probe_model = fresh_model()
head_keys = ["classifier", "score", "pre_classifier"]
for name, param in probe_model.named_parameters():
    if not any(h in name for h in head_keys):
        param.requires_grad = False
count_trainable_params(probe_model)

probe_trainer, probe_metrics = run_training(
    "linear_probe",
    probe_model,
    tokenized["train"],
    tokenized[split_val],
    probe_args,
)


In [ ]:
if get_peft_model is None:
    print("PEFT not installed; run `pip install peft` to enable LoRA")
else:
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=cfg["lora_r"],
        lora_alpha=cfg["lora_alpha"],
        lora_dropout=cfg["lora_dropout"],
        target_modules=["q_lin", "k_lin", "v_lin", "out_lin", "query", "key", "value", "dense"],
        bias="none",
    )

    lora_model = get_peft_model(fresh_model(), lora_config)
    lora_model.print_trainable_parameters()

    lora_args = TrainingArguments(
        output_dir="outputs/lora",
        learning_rate=2e-4,
        per_device_train_batch_size=cfg["batch_size"],
        per_device_eval_batch_size=cfg["eval_batch_size"],
        num_train_epochs=cfg["num_epochs_lora"],
        weight_decay=0.0,
        evaluation_strategy="epoch",
        save_strategy="no",
        logging_steps=100,
        warmup_ratio=cfg["warmup_ratio"],
        seed=cfg["seed"],
        report_to="none",
    )

    lora_trainer, lora_metrics = run_training(
        "lora",
        lora_model,
        tokenized["train"],
        tokenized[split_val],
        lora_args,
    )


In [ ]:
results
